# 🤖 Fake Radar — Fine-tuning de BETO (BERT en Español)

**Responsable:** B. Aguayo — Ingeniería IA / NLP  
**Proyecto:** Fake Radar: Sistema Inteligente de Detección de Desinformación  
**Materia:** Inteligencia Artificial — SCC-1002  
**Instituto Tecnológico de Tijuana — Enero-Junio 2026**

---

## ¿Qué hace este notebook?

Entrena el modelo **BETO** (`dccuchile/bert-base-spanish-wwm-uncased`) sobre el dataset real de noticias en español (`train.xlsx`, 676 noticias balanceadas), para clasificar si una noticia es **Fake** o **Verdadera**.

### ¿Qué es BETO?
BETO es BERT entrenado en español por la Universidad de Chile sobre el corpus Wikipedia en español. Al hacer **fine-tuning**, tomamos ese modelo que ya entiende español y lo especializamos en detectar desinformación con nuestros datos.

### Pipeline:
```
train.xlsx (español)
    → Limpieza y preprocesamiento
    → Tokenización con BETO tokenizer
    → Fine-tuning (3 épocas)
    → Evaluación (accuracy, F1, matriz de confusión)
    → Modelo guardado → integración con API backend
```

### Nota de hardware:
Este notebook está optimizado para CPU (8GB RAM, 4 núcleos). El entrenamiento toma entre 30 y 90 minutos dependiendo del equipo. Puede dejarse corriendo sin supervisión.

---

## Bloque 1 — Instalar dependencias

Ejecutar una sola vez. Si ya instalaste antes, puedes saltar este bloque.

In [1]:
import subprocess, sys

def instalar(paquete):
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', paquete,
        '--user', '-q', '--no-warn-script-location'
    ])

paquetes = ['transformers', 'torch', 'accelerate', 'scikit-learn', 'pandas', 'openpyxl', 'matplotlib', 'seaborn']

for p in paquetes:
    try:
        __import__(p.split('[')[0].replace('-','_'))
        print(f'  OK: {p}')
    except ImportError:
        print(f'  Instalando: {p}...')
        instalar(p)
        print(f'  Listo: {p}')

print('\nTodas las dependencias están listas.')

  OK: transformers
  OK: torch
  OK: accelerate
  Instalando: scikit-learn...
  Listo: scikit-learn
  OK: pandas
  OK: openpyxl
  OK: matplotlib
  OK: seaborn

Todas las dependencias están listas.


## Bloque 2 — Importar librerías

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # necesario para guardar graficas sin pantalla
import seaborn as sns
import re
import os
import pickle
import warnings
import time
warnings.filterwarnings('ignore')

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score
)

# Configuracion de dispositivo
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo de entrenamiento: {DEVICE}')
if DEVICE.type == 'cpu':
    print('  Usando CPU — el entrenamiento tomara entre 30 y 90 minutos.')
    print('  Puedes dejar la laptop corriendo y revisar el resultado al despertar.')
else:
    print('  GPU detectada — el entrenamiento sera rapido (5-15 min).')

# Semilla global para reproducibilidad
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Colores del proyecto
COLOR_FAKE = '#E74C3C'
COLOR_REAL = '#2ECC71'

print('Librerias importadas correctamente.')

Dispositivo de entrenamiento: cpu
  Usando CPU — el entrenamiento tomara entre 30 y 90 minutos.
  Puedes dejar la laptop corriendo y revisar el resultado al despertar.
Librerias importadas correctamente.


## Bloque 3 — Cargar y limpiar el dataset

Dataset: `train.xlsx` — 676 noticias en español (338 Fake, 338 True), perfectamente balanceado.

In [3]:
# Cargar el archivo — debe estar en la misma carpeta que este notebook
DATASET_PATH = './train.xlsx'

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f'No se encontro {DATASET_PATH}\n'
        'Asegurate de que train.xlsx este en la misma carpeta que este notebook.'
    )

df_raw = pd.read_excel(DATASET_PATH)
print(f'Dataset cargado: {len(df_raw)} filas, columnas: {list(df_raw.columns)}')

# Renombrar columnas y mapear etiquetas
df = df_raw.rename(columns={
    'Category': 'label_raw',
    'Headline': 'title',
    'Text': 'text'
}).copy()

# 0 = Fake, 1 = Verdadera
df['label'] = df['label_raw'].map({'Fake': 0, 'True': 1})
df = df.dropna(subset=['label', 'title', 'text']).reset_index(drop=True)
df['label'] = df['label'].astype(int)

def limpiar_texto(t):
    """Limpieza basica del texto conservando el contenido en espanol."""
    if not isinstance(t, str):
        return ''
    t = re.sub(r'http\S+', '', t)          # eliminar URLs
    t = re.sub(r'<.*?>', '', t)            # eliminar HTML
    t = re.sub(r'\*NUMBER\*', '', t)       # eliminar placeholder
    t = re.sub(r'\s+', ' ', t)            # normalizar espacios
    return t.strip()

df['title_clean'] = df['title'].apply(limpiar_texto)
df['text_clean']  = df['text'].apply(limpiar_texto)

# Combinar titulo + cuerpo. Truncar a 400 palabras (BETO acepta max ~512 tokens)
df['texto_completo'] = (df['title_clean'] + ' ' + df['text_clean']).apply(
    lambda x: ' '.join(x.split()[:400])
)

print(f'\nDistribucion final:')
print(f'  Fake (0): {(df.label==0).sum()} noticias')
print(f'  True (1): {(df.label==1).sum()} noticias')
print(f'  Total:    {len(df)} noticias')
print(f'  Longitud promedio: {df.texto_completo.apply(lambda x: len(x.split())).mean():.0f} palabras')
print(f'\nEjemplo FAKE: "{df[df.label==0]["titulo"].iloc[0][:90]}..."' if 'titulo' in df.columns else '')
print(f'Ejemplo FAKE: "{df[df.label==0]["title_clean"].iloc[0][:90]}"')
print(f'Ejemplo REAL: "{df[df.label==1]["title_clean"].iloc[0][:90]}"')

Dataset cargado: 676 filas, columnas: ['Id', 'Category', 'Topic', 'Source', 'Headline', 'Text', 'Link']

Distribucion final:
  Fake (0): 338 noticias
  True (1): 338 noticias
  Total:    676 noticias
  Longitud promedio: 306 palabras

Ejemplo FAKE: "RAE INCLUIRÁ LA PALABRA "LADY" EN EL DICCIONARIO DEL IDIOMA ESPAÑOL COMO DEFINICIÓN DE "MU"
Ejemplo REAL: "UNAM capacitará a maestros para aprobar prueba Pisa"


## Bloque 4 — Dividir en entrenamiento y prueba

In [4]:
X = df['texto_completo'].astype(str).to_numpy()
y = df['label'].astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print('Division del dataset:')
print(f'  Entrenamiento: {len(X_train)} noticias (80%)')
print(f'    FAKE: {(y_train==0).sum()} | REAL: {(y_train==1).sum()}')
print(f'  Prueba:        {len(X_test)} noticias (20%)')
print(f'    FAKE: {(y_test==0).sum()} | REAL: {(y_test==1).sum()}')

Division del dataset:
  Entrenamiento: 540 noticias (80%)
    FAKE: 270 | REAL: 270
  Prueba:        136 noticias (20%)
    FAKE: 68 | REAL: 68


## Bloque 5 — Tokenización con BETO

El tokenizador convierte el texto en secuencias de números que BETO entiende. `max_length=256` es suficiente para la mayoría de noticias y reduce el uso de memoria comparado con 512.

In [5]:
MODELO_BETO = 'dccuchile/bert-base-spanish-wwm-uncased'
MAX_LEN = 256

print(f'Descargando tokenizador BETO ({MODELO_BETO})...')
print('(La primera vez descarga ~500MB — requiere internet)')

tokenizador = BertTokenizerFast.from_pretrained(MODELO_BETO)

print('Tokenizador cargado.')
print(f'Vocabulario: {tokenizador.vocab_size:,} tokens en espanol')

# Prueba rapida
ejemplo = tokenizador('Esta noticia es completamente falsa y sensacionalista',
                      return_tensors='pt', truncation=True, max_length=MAX_LEN)
print(f'Ejemplo tokenizado: {ejemplo["input_ids"].shape[1]} tokens')

Descargando tokenizador BETO (dccuchile/bert-base-spanish-wwm-uncased)...
(La primera vez descarga ~500MB — requiere internet)


tokenizer_config.json:   0%|          | 0.00/310 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Tokenizador cargado.
Vocabulario: 31,002 tokens en espanol
Ejemplo tokenizado: 10 tokens


## Bloque 6 — Crear Dataset de PyTorch

PyTorch necesita que los datos estén en un formato especial (`Dataset`) para manejar los lotes de entrenamiento de forma eficiente.

In [6]:
class NoticiasDataset(Dataset):
    """
    Dataset personalizado para noticias.
    Tokeniza el texto al vuelo para ahorrar memoria.
    """
    def __init__(self, textos, etiquetas, tokenizador, max_len):
        self.textos     = textos
        self.etiquetas  = etiquetas
        self.tokenizador = tokenizador
        self.max_len    = max_len

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        encoding = self.tokenizador(
            str(self.textos[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.etiquetas[idx], dtype=torch.long)
        }

# Crear datasets
BATCH_SIZE = 8  # 8 es seguro para 8GB RAM con CPU

train_dataset = NoticiasDataset(X_train, y_train, tokenizador, MAX_LEN)
test_dataset  = NoticiasDataset(X_test,  y_test,  tokenizador, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Dataset de entrenamiento: {len(train_dataset)} muestras, {len(train_loader)} lotes')
print(f'Dataset de prueba:        {len(test_dataset)} muestras, {len(test_loader)} lotes')
print(f'Tamano de lote: {BATCH_SIZE}')

Dataset de entrenamiento: 540 muestras, 68 lotes
Dataset de prueba:        136 muestras, 17 lotes
Tamano de lote: 8


## Bloque 7 — Cargar modelo BETO preentrenado

Cargamos BETO con una capa de clasificación de 2 clases (Fake / Real) encima.

In [7]:
print(f'Cargando modelo BETO para clasificacion binaria...')

modelo = BertForSequenceClassification.from_pretrained(
    MODELO_BETO,
    num_labels=2,
    ignore_mismatched_sizes=True
)
modelo = modelo.to(DEVICE)

total_params = sum(p.numel() for p in modelo.parameters())
print(f'Modelo cargado en {DEVICE}.')
print(f'Parametros totales: {total_params:,}')
print(f'(BERT base tiene ~110 millones de parametros)')

Cargando modelo BETO para clasificacion binaria...


config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; no

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Modelo cargado en cpu.
Parametros totales: 109,852,418
(BERT base tiene ~110 millones de parametros)


## Bloque 8 — Configurar optimizador y scheduler

- **AdamW**: variante de Adam recomendada para transformers, con regularización de pesos
- **Scheduler lineal con warmup**: la tasa de aprendizaje sube gradualmente al inicio y baja al final, lo que estabiliza el fine-tuning

In [8]:
EPOCHS       = 3
LEARNING_RATE = 2e-5  # tasa estandar para fine-tuning BERT
WARMUP_STEPS  = int(len(train_loader) * 0.1)  # 10% del primer epoch
TOTAL_STEPS   = len(train_loader) * EPOCHS

optimizador = AdamW(modelo.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

scheduler = get_linear_schedule_with_warmup(
    optimizador,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_STEPS
)

print(f'Configuracion de entrenamiento:')
print(f'  Epocas:          {EPOCHS}')
print(f'  Learning rate:   {LEARNING_RATE}')
print(f'  Pasos de warmup: {WARMUP_STEPS}')
print(f'  Pasos totales:   {TOTAL_STEPS}')
print(f'  Lotes por epoca: {len(train_loader)}')

Configuracion de entrenamiento:
  Epocas:          3
  Learning rate:   2e-05
  Pasos de warmup: 6
  Pasos totales:   204
  Lotes por epoca: 68


## Bloque 9 — Entrenamiento

**Este bloque tarda entre 30 y 90 minutos en CPU. Dejarlo corriendo.**

El progreso se imprime al final de cada época mostrando la pérdida y la precisión en el conjunto de prueba.

In [ ]:
def evaluar(modelo, loader, device):
    """Evalua el modelo sobre un DataLoader y devuelve accuracy y F1."""
    modelo.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = modelo(input_ids=input_ids, attention_mask=attention_mask)
            preds   = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='weighted')
    return acc, f1, all_labels, all_preds


# Historial para graficar despues
historial = {'epoch': [], 'loss': [], 'accuracy': [], 'f1': []}

mejor_accuracy = 0.0
inicio_total   = time.time()

print('=' * 60)
print('  INICIO DEL ENTRENAMIENTO BETO')
print(f'  {EPOCHS} epocas x {len(train_loader)} lotes = {TOTAL_STEPS} pasos')
print('=' * 60)

for epoch in range(1, EPOCHS + 1):
    inicio_epoch = time.time()
    modelo.train()
    total_loss = 0.0

    for i, batch in enumerate(train_loader, 1):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        optimizador.zero_grad()
        outputs = modelo(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()

        # Gradient clipping: evita que los gradientes exploten
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)

        optimizador.step()
        scheduler.step()

        total_loss += loss.item()

        # Progreso cada 10 lotes
        if i % 10 == 0 or i == len(train_loader):
            print(f'  Epoca {epoch}/{EPOCHS} | Lote {i}/{len(train_loader)} | Loss: {total_loss/i:.4f}', end='\r')

    # Evaluacion al final de cada epoca
    avg_loss = total_loss / len(train_loader)
    acc, f1, _, _ = evaluar(modelo, test_loader, DEVICE)
    elapsed = (time.time() - inicio_epoch) / 60

    historial['epoch'].append(epoch)
    historial['loss'].append(avg_loss)
    historial['accuracy'].append(acc)
    historial['f1'].append(f1)

    print(f'\n  Epoca {epoch}/{EPOCHS} completada en {elapsed:.1f} min')
    print(f'    Loss promedio: {avg_loss:.4f}')
    print(f'    Accuracy:      {acc*100:.2f}%')
    print(f'    F1-Score:      {f1:.4f}')

    # Guardar el mejor modelo
    if acc > mejor_accuracy:
        mejor_accuracy = acc
        modelo.save_pretrained('./beto_fakenews_model')
        tokenizador.save_pretrained('./beto_fakenews_model')
        print(f'    Mejor modelo guardado (accuracy: {acc*100:.2f}%)')

total_min = (time.time() - inicio_total) / 60
print()
print('=' * 60)
print(f'  ENTRENAMIENTO COMPLETADO en {total_min:.1f} minutos')
print(f'  Mejor accuracy en prueba: {mejor_accuracy*100:.2f}%')
print('=' * 60)

  INICIO DEL ENTRENAMIENTO BETO
  3 epocas x 68 lotes = 204 pasos


## Bloque 10 — Evaluación final y gráficas

In [ ]:
# Cargar el mejor modelo guardado
mejor_modelo = BertForSequenceClassification.from_pretrained('./beto_fakenews_model')
mejor_modelo = mejor_modelo.to(DEVICE)

acc_final, f1_final, y_true, y_pred = evaluar(mejor_modelo, test_loader, DEVICE)

print('REPORTE FINAL DEL MODELO BETO')
print('=' * 55)
print(f'Accuracy: {acc_final*100:.2f}%')
print(f'F1-Score: {f1_final:.4f}')
print()
print(classification_report(y_true, y_pred, target_names=['FAKE', 'REAL']))

In [ ]:
# ============================================================
# Grafica 1: Curvas de entrenamiento
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Curvas de Entrenamiento — BETO Fine-tuning', fontsize=14, fontweight='bold')

epocas = historial['epoch']

axes[0].plot(epocas, historial['loss'], 'o-', color='#E74C3C', linewidth=2, markersize=8)
axes[0].set_title('Perdida por epoca', fontsize=12)
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Loss')
axes[0].set_xticks(epocas)
for x, y_val in zip(epocas, historial['loss']):
    axes[0].annotate(f'{y_val:.3f}', (x, y_val), textcoords='offset points', xytext=(0,10), ha='center')

axes[1].plot(epocas, [a*100 for a in historial['accuracy']], 'o-', color='#2ECC71', linewidth=2, markersize=8, label='Accuracy')
axes[1].plot(epocas, [f*100 for f in historial['f1']], 's--', color='#3498DB', linewidth=2, markersize=8, label='F1-Score')
axes[1].set_title('Accuracy y F1 por epoca', fontsize=12)
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Valor (%)')
axes[1].set_xticks(epocas)
axes[1].legend()
axes[1].set_ylim(0, 110)
for x, y_val in zip(epocas, historial['accuracy']):
    axes[1].annotate(f'{y_val*100:.1f}%', (x, y_val*100), textcoords='offset points', xytext=(0,10), ha='center', color='#2ECC71')

plt.tight_layout()
plt.savefig('./plot_07_curvas_entrenamiento.png', dpi=150, bbox_inches='tight')
plt.close()
print('Grafica guardada: plot_07_curvas_entrenamiento.png')

In [ ]:
# ============================================================
# Grafica 2: Matriz de confusion final
# ============================================================
from sklearn.metrics import ConfusionMatrixDisplay, precision_score, recall_score

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Evaluacion Final — BETO (BERT en Espanol)', fontsize=14, fontweight='bold')

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['FAKE', 'REAL'])
disp.plot(ax=axes[0], colorbar=False, cmap='RdYlGn')
axes[0].set_title('Matriz de Confusion', fontsize=12)

metricas = {
    'Accuracy':  acc_final,
    'Precision': precision_score(y_true, y_pred, average='weighted'),
    'Recall':    recall_score(y_true, y_pred, average='weighted'),
    'F1-Score':  f1_final,
}
colores_b = ['#2ECC71' if v >= 0.80 else '#E74C3C' for v in metricas.values()]
bars = axes[1].bar(metricas.keys(), metricas.values(), color=colores_b, edgecolor='white', width=0.5)
for bar, val in zip(bars, metricas.values()):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.1%}', ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, 1.15)
axes[1].set_title('Metricas de Evaluacion', fontsize=12)
axes[1].axhline(y=0.80, color='gray', linestyle='--', alpha=0.5, label='Umbral 80%')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('./plot_08_evaluacion_beto.png', dpi=150, bbox_inches='tight')
plt.close()
print('Grafica guardada: plot_08_evaluacion_beto.png')

## Bloque 11 — Probar con noticias nuevas

In [ ]:
def predecir(texto, modelo, tokenizador, device, max_len=256):
    """
    Clasifica una noticia como FAKE o REAL usando BETO.
    Devuelve la prediccion y la probabilidad de confianza.
    """
    modelo.eval()
    encoding = tokenizador(
        texto,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    with torch.no_grad():
        outputs = modelo(
            input_ids=encoding['input_ids'].to(device),
            attention_mask=encoding['attention_mask'].to(device)
        )
    probs = torch.softmax(outputs.logits, dim=1).squeeze().cpu().numpy()
    pred  = int(np.argmax(probs))
    etiqueta = 'REAL' if pred == 1 else 'FAKE'
    return etiqueta, probs[pred] * 100, probs


print('PRUEBAS DEL MODELO BETO CON NOTICIAS NUEVAS')
print('=' * 60)

noticias_prueba = [
    ("AMLO confirmó que regalará un millón de pesos a cada mexicano para Navidad, según fuentes anónimas",
     "noticias sensacionalistas"),
    ("La UNAM anuncia nuevo programa de becas para estudiantes de posgrado en ciencias exactas",
     "noticia institucional"),
    ("¡URGENTE! Científicos DESCUBREN que el agua del grifo causa cáncer — COMPARTE antes de que lo borren",
     "fake news tipica"),
    ("El Banco de México reportó una inflacion del 4.2% anual en su informe trimestral de politica monetaria",
     "noticia economica real"),
]

for titulo, tipo in noticias_prueba:
    etiqueta, confianza, probs = predecir(titulo, mejor_modelo, tokenizador, DEVICE)
    emoji = 'ALERTA' if etiqueta == 'FAKE' else 'OK'
    print(f'[{emoji}] {etiqueta} ({confianza:.1f}% confianza)')
    print(f'  Tipo:   {tipo}')
    print(f'  Titulo: "{titulo[:70]}"')
    print(f'  P(FAKE)={probs[0]*100:.1f}%  P(REAL)={probs[1]*100:.1f}%')
    print()

## Bloque 12 — Guardar modelo para integración con API

In [ ]:
# El modelo ya fue guardado en formato HuggingFace durante el entrenamiento
# en la carpeta ./beto_fakenews_model/
# Esto es lo que el equipo de backend carga en la API

print('El modelo BETO esta guardado en: ./beto_fakenews_model/')
print()
print('Archivos generados:')
if os.path.exists('./beto_fakenews_model'):
    for f in os.listdir('./beto_fakenews_model'):
        size_kb = os.path.getsize(f'./beto_fakenews_model/{f}') / 1024
        print(f'  {f}: {size_kb:.0f} KB')

print()
print('Codigo para cargar en el backend (para Casildo / Jaime / Aboytes):')
print('''
  from transformers import BertTokenizerFast, BertForSequenceClassification
  import torch

  tokenizador = BertTokenizerFast.from_pretrained('./beto_fakenews_model')
  modelo      = BertForSequenceClassification.from_pretrained('./beto_fakenews_model')
  modelo.eval()

  def analizar(texto):
      enc  = tokenizador(texto, return_tensors="pt", truncation=True, max_length=256)
      with torch.no_grad():
          out = modelo(**enc)
      probs = torch.softmax(out.logits, dim=1).squeeze().tolist()
      pred  = "FAKE" if probs[0] > probs[1] else "REAL"
      return {"prediccion": pred, "p_fake": probs[0], "p_real": probs[1]}
''')

print()
print('Metricas finales del modelo:')
print(f'  Modelo:   BETO (dccuchile/bert-base-spanish-wwm-uncased)')
print(f'  Dataset:  train.xlsx — 676 noticias en espanol')
print(f'  Accuracy: {acc_final*100:.2f}%')
print(f'  F1-Score: {f1_final:.4f}')

---

## Resumen del sprint

| Artefacto | Descripcion |
|-----------|-------------|
| `beto_fakenews_model/` | Modelo BETO fine-tuned listo para produccion |
| `plot_07_curvas_entrenamiento.png` | Curvas de loss y accuracy por epoca |
| `plot_08_evaluacion_beto.png` | Matriz de confusion y metricas finales |
| `train.xlsx` | Dataset original en espanol utilizado |

---
*Fake Radar — Instituto Tecnológico de Tijuana — Enero-Junio 2026*